In [18]:
import torch


In [ ]:
import math
import torch.nn as nn


In [ ]:
text = ""
with open("file.txt", 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        content = line.split(sep = '-', maxsplit=1)
        if '<Media omitted>' in content:
            continue
        if len(content)<2:
            text += "\n" + line
            continue
        text += "\n"
        text += content[1].strip()

chars = list(set(text))
chars.sort()
stoi = {char : ind for ind, char in enumerate(chars)}
itos = {ind : char for ind, char in enumerate(chars)}

def encode(text):
    return [stoi[char] for char in text]

def decode(embedding):
    return [itos[ind] for ind in embedding]

def train_test_split(data, train=0.9):
    n = int(train*len(data))
    tr = data[:n]
    val = data[n:len(data)]
    return tr, val


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
train_data, val_data = train_test_split(data)

torch.manual_seed(42)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split=='train' else val_data
    inds = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in inds])
    y = torch.stack([data[i+1:i+block_size+1] for i in inds])
    return x, y

xb, yb = get_batch('train')
print(xb)
print(yb)


In [ ]:
for i in range(batch_size):
    for j in range(block_size):
        context = xb[i, :j+1]
        target = yb[i, j]


In [ ]:
class AdamOptimizer:
    def __init__(self, params, lr=1e-3, eps=1e-8, betas=(0.9, 0.999)):
        self.params = list(params)
        self.lr = lr
        self.eps = eps
        self.beta1 = betas[0]
        self.beta2 = betas[1]
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue
                self.m[i] = self.beta1*self.m[i] + (1-self.beta1)*p.grad
                m_hat = self.m[i]/(1-self.beta1**self.t)
                self.v[i] = self.beta2*self.v[i] + (1-self.beta2)*(p.grad**2)
                v_hat = self.v[i]/(1-self.beta2**self.t)
                p -= (m_hat*self.lr)/(v_hat.sqrt()+self.eps)
    
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


In [37]:
def softmax(x):
  exp_x = torch.exp(x)
  return exp_x/exp_x.sum(dim=-1, keepdim=True) # why -1? -> the matrix is batchxqueryxkey, so we have to softmax over all keys

def causal_softmax(x): # matrix = (batch, block_size, block_size) - attention matrix
    mask = torch.tril(torch.ones(block_size, block_size))
    x_mask = x.masked_fill(mask == 0, -float('inf'))
    return softmax(x_mask)  


In [ ]:
n_embd = 32
d_k = n_embd # Dimension of q, k, v output
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.W_q = nn.Linear(n_embd, d_k, bias=False)
        self.W_k = nn.Linear(n_embd, d_k, bias=False)
        self.W_v = nn.Linear(n_embd, d_k, bias=False)
    
    def forward(self, batch): # batch = (batch_size, block_size, embedding_size), W_q = (embedding_size, output_size=d_k)
        Q = self.W_q(batch) # (batch_size, block_size, d_k)
        K = self.W_k(batch)
        V = self.W_v(batch)
        w1 = Q@K.transpose(-2, -1) # we don't transpose each block in a batch, only within each block
        w2 = w1/math.sqrt(d_k)
        W = causal_softmax(w2) # (batch_size, block_size, block_size)
        new_tokens = W @ V
        return new_tokens


In [ ]:
def relu(x): return torch.clamp(x, min=0)
def gelu(x): return x*0.5*(1+torch.erf(x/math.sqrt(2)))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 =  nn.Linear(n_embd, 4*n_embd, bias=True)
        self.l2 = nn.Linear(4*n_embd, n_embd, bias=True)

    def forward(self, batch):
        x1 = self.l1(batch)
        x2 = gelu(x1)
        x3 = self.l2(x2)
        return x3


In [ ]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = Attention()
        self.attention_norm = nn.LayerNorm(n_embd)
        self.ff = FeedForward()
        self.ff_norm = nn.LayerNorm(n_embd)
    
    def forward(self, batch):
        batch = batch + self.attention(self.attention_norm(batch)) # nn.Module runs self.forward automatically
        batch = batch + self.ff(self.ff_norm(batch))
        return batch
    